<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG201 - Aprendizaje supervisado**</span>

Bienvenido al curso!!!

Vamos a revisar diferentes algoritmos de <span style="color:gold">aprendizaje supervisado</span> y su aplicación en Geología. <br>
Es necesario que tengas un conocimiento previo en programación con Python, álgebra lineal, estadística y geología.


<span style="color:gold; font-size:20px">**Regresión Logística**</span>
***

- [¿Qué es la Regresión Logística?](#parte-1)
- [Caso de estudio: Mapeo litológico de la superficie lunar](#parte-2)
- [Generación de mapas de probabilidades](#parte-3)
- [En conclusión...](#parte-4)
***

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'rasterio'])

<a id="parte-1"></a>

### <span style="color:lightgreen">**¿Qué es la regresión logística?**</span>
***

La regresión logística es un modelo estadístico utilizado para modelar la relación entre una <span style="color:#43c6ac">variable dependiente categórica</span> (también conocida como variable objetivo o variable de respuesta) y una o más variables independientes (también llamadas variables predictoras o características). 

> A diferencia de la regresión lineal, que se usa para predecir valores continuos, la regresión logística se utiliza principalmente para predecir probabilidades y clasificaciones en variables categóricas.

En su aplicación más común, la <span style="color:#43c6ac">regresión logística binaria</span> se utiliza para resolver problemas de clasificación donde la variable objetivo tiene solo dos posibles resultados, como "sí/no" o "0/1". Sin embargo, también se puede adaptar para situaciones en las que la variable objetivo tiene más de dos categorías. En estos casos, se emplea la <span style="color:#43c6ac">regresión logística multinomial</span>, que permite clasificar observaciones en múltiples clases manteniendo el enfoque probabilístico.

La fórmula del modelo logístico para una variable dependiente binaria es:



$$
\Large P(y = 1) = \frac{1}{1 + e^{-(b_0 + b_1x_1 + b_2x_2 + \ldots + b_nx_n)}}
$$



Donde:
- $P(y = 1)\,$ es la probabilidad de que ocurra el evento de interés (por ejemplo, que $\,y=1$).
- $b_0\,$ es el intercepto del modelo (el término de sesgo)..
- $b_1, b_2,..., b_n$ son los coeficientes que se ajustan durante el entrenamiento.
- $x_1, x_2,..., x_n$ son las variables independientes (o características).

La sigmoide transforma el puntaje lineal $z$ en un valor entre 0 y 1. Con $z=0$, obtenemos $p=0.5$. Al aumentar $z$, aumenta $p$, pero un cambio fijo de $z$ produce un cambio distinto de $p$ según el punto de partida.

El modelo ajusta sus coeficientes minimizando la pérdida logarítmica con una penalización. Esta pérdida aumenta si el modelo asigna poca probabilidad a la clase observada. `predict_proba` conserva las dos columnas del modelo. `predict` escoge la clase con mayor probabilidad estimada.

***
<span style="color:gold">**Ventajas y limitaciones de la Regresión Logística** </span>

- <span style="color:lightgreen">Interpretabilidad:</span> los coeficientes pueden interpretarse en términos de cómo se relacionan con el logaritmo de los odds de pertenecer a una clase específica.
- <span style="color:lightgreen">Flexibilidad:</span> aunque se usa principalmente para clasificación binaria, puede extenderse a más clases y manejar tanto variables continuas como categóricas.
- <span style="color:lightgreen">Probabilidades directas:</span> proporciona no solo una clasificación, sino también una probabilidad estimada cuya calibración podemos evaluar en datos representativos.
- <span style="color:orange">Linealidad en el logit:</span> asume que la relación entre las variables independientes y el logit (logaritmo de los odds, ln(p/(1−p))) es lineal.
- <span style="color:orange">No captura relaciones complejas:</span> Puede no ser adecuada para relaciones complejas no lineales entre variables.

***

<a id="parte-2"></a>

### <span style="color:lightgreen">**Caso de estudio: Mapeo litológico de la superficie lunar** </span>
***

El estudio de la litología de la superficie lunar proporciona información crucial sobre los procesos geológicos que han dado forma a nuestro satélite natural. El área específica de estudio se encuentra alrededor del Mare Crisium y estos rasters provienen del trabajo realizado por [Bhatt et al (2019)](https://zenodo.org/records/5762834).


    Entrenaremos con anortosita y basalto toleítico terrestres, y llevaremos el modelo a los píxeles de Mare Crisium. Es una prueba de **transferencia de dominio**. Las clases y la escala de las entradas deben tener sentido en las dos fuentes. Necesitaríamos referencias lunares independientes para medir el desempeño del mapa.

***
<span style="color:gold">**Procesamiento y exploración de datos** </span>

Empezaremos cargando los rasters de Ca, Fe, Mg de la superficie lunar y las tablas geoquímicas de anortositas y basaltos toleíticos de la Tierra:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

Leemos las tres capas originales y comprobamos su alineación. Así la lección puede ejecutarse de forma independiente.

In [ ]:
# Abrir las tres capas y conservar su referencia espacial.
capas = {}
for elemento in ("Ca", "Fe", "Mg"):
    ruta = f"files/rasters/lunar_mapping/raster_LPGRS_{elemento}.tif"
    with rasterio.open(ruta) as archivo:
        datos = archivo.read(1, masked=True).astype(float).filled(np.nan)
        grilla = (archivo.shape, archivo.transform, archivo.crs)
        capas[elemento] = datos
        if elemento == "Ca":
            grilla_referencia = grilla
            limites = archivo.bounds
            perfil = archivo.profile
        elif grilla != grilla_referencia:
            raise ValueError(f"La grilla de {elemento} difiere de la de Ca.")

raster_Ca, raster_Fe, raster_Mg = (capas[e] for e in ("Ca", "Fe", "Mg"))
extension = (limites.left, limites.right, limites.bottom, limites.top)
validos = np.isfinite(raster_Ca) & np.isfinite(raster_Fe) & np.isfinite(raster_Mg)

En la imagen de falso color asignamos Ca al rojo, Fe al verde y Mg al azul. El escalado de percentiles se usa solo en la figura. El clasificador recibe las concentraciones originales.

In [ ]:
rgb = np.zeros((*raster_Ca.shape, 3))
for canal, raster in enumerate(capas.values()):
    bajo, alto = np.quantile(raster[validos], [0.02, 0.98])
    rgb[validos, canal] = np.clip((raster[validos] - bajo) / (alto - bajo), 0, 1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(rgb, extent=extension)
ax.set(title="Mare Crisium · Ca / Fe / Mg", xlabel="Longitud (°)", ylabel="Latitud (°)")
plt.show()

Ahora abrimos las tablas terrestres. Cada fila contiene medidas de una muestra y su litología de referencia.

In [ ]:
# Leer los datos de geoquímica terrestre
anorth = pd.read_csv("files/anorthosite.csv")
basalt = pd.read_csv("files/tholeitic_basalt.csv")

# Agregar las columnas de litología
anorth["target"] = "anorthosite"
basalt["target"] = "basalt"

Ahora, mostraremos un resumen estadístico de ambas litologías:

In [ ]:
# Resumen estadístico de anortosita
anorth.describe().T

In [ ]:
# Resumen estadístico de basalto
basalt.describe().T

Podemos observar un desbalance en la cantidad de datos entre anortositas (58) y basaltos (1381). 

Para investigar si existe un patrón de separación definido entre ambas litologías, visualizaremos la distribución de las muestras en una figura 3D.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(10, 3))
for eje, elemento in zip(ejes, ["Ca", "Fe", "Mg"]):
    eje.boxplot([anorth[elemento].dropna(), basalt[elemento].dropna()])
    eje.set(xticks=[1, 2], xticklabels=["Anortosita", "Basalto"], ylabel=f"{elemento} (%)")
fig.tight_layout()
plt.show()

En el gráfico 3D se aprecia una clara separación entre las muestras de anortositas y basaltos. El modelo deberá aprender a distinguir entre estas clases utilizando las variables Ca, Fe y Mg para predecir la probabilidad de pertenencia a una litología específica.

In [ ]:
# Unir ambas tablas
data = pd.concat([anorth, basalt], ignore_index=True)
data = data.loc[np.isfinite(data[["Ca", "Fe", "Mg"]]).all(axis=1)]

# Mostrar el conteo de datos disponibles
conteo = data["target"].value_counts()
for r, c in conteo.items():
    print(f"Litología: {r:<11} | Datos disponibles: {c:<4} ({c / len(data):.2%})")

Nuestro modelo utilizará las concentraciones de `Ca`, `Fe` y `Mg` como características (features) para predecir la variable objetivo (`target`), que representa la clasificación litológica entre basalto y anortosita.

In [ ]:
X = data[["Ca", "Fe", "Mg"]].values  # Features
y = data["target"].values            # Target

Utilizaremos la función `train_test_split` para dividir los datos en conjuntos de entrenamiento y prueba:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=123, stratify=y
)

Balanceamos únicamente el entrenamiento. Primero contamos cada clase, luego agregamos exactamente las filas que faltan mediante un muestreo con reemplazo. La prueba conserva su proporción inicial.

In [ ]:
clases, conteos = np.unique(y_train, return_counts=True)
minoritaria = clases[conteos.argmin()]
n_adicionales = conteos.max() - conteos.min()
rng = np.random.default_rng(42)
copias = rng.choice(np.flatnonzero(y_train == minoritaria), size=n_adicionales, replace=True)
X_balanceado = np.concatenate([X_train, X_train[copias]])
y_balanceado = np.concatenate([y_train, y_train[copias]])
print(pd.Series(y_balanceado).value_counts())

Instanciaremos el modelo usando la función `LogisticRegression` y usaremos el hiperparámetro `C` para regularizar el entrenamiento del modelo.

> La <span style="color:#43c6ac">regularización</span> es una técnica utilizada para prevenir el sobreajuste del modelo al penalizar los coeficientes grandes, lo que ayuda a mantener el modelo más simple y generalizable a nuevos datos.
> 
> `C` es el inverso de la fuerza de regularización, la regresión logística en scikit-learn aplica por defecto una regularización de tipo L2 (Ridge), que penaliza la suma de los cuadrados de los coeficientes del modelo.

In [ ]:
# Crear y entrenar el modelo de regresión logística
model = LogisticRegression(C=1e-4)
model.fit(X_balanceado, y_balanceado)

Vamos a evaluar el rendimiento del modelo de clasificación usando la función `classification_report`:

> Esta función genera un reporte que incluye varias métricas de evaluación comunes para cada clase en el conjunto de datos. 

In [ ]:
# Hacer predicciones
y_pred = model.predict(X_test)

# Evaluar el modelo
print(classification_report(y_test, y_pred))

El reporte compara predicciones con etiquetas **terrestres** de prueba. Compara el recall de ambas clases. Si la prueba contiene réplicas de las mismas localidades, una división por localidad aportaría evidencia más útil para otra campaña. Sobremuestrear cambia la prevalencia vista por el modelo y puede alterar la calibración de sus probabilidades.

***
<span style="color:gold">**Análisis de coeficientes** </span>

Podemos obtener los coeficientes del modelo de Regresión Logística usando el atributo `coef_`:

In [ ]:
# Coeficientes del modelo
coefficients = model.coef_[0]

print("Coeficientes del modelo")
print("-----------------------")
for feature, coef in zip(["Ca", "Fe", "Mg"], coefficients):
    print(f"{feature}: {coef:>6.3f}")

Para una probabilidad $p$, los **odds** son $p/(1-p)$. El logaritmo de los odds para `model.classes_[1]` es el puntaje lineal. Si Ca aumenta un punto porcentual y Fe y Mg se mantienen fijos, los odds se multiplican por $\exp(b_{Ca})$.

Duplicar los odds transforma $p=0.20$ en aproximadamente $0.33$. Si partimos de $p=0.50$, pasa a aproximadamente $0.67$. El multiplicador de los odds sirve en ambos casos, pero el cambio de probabilidad es distinto.

In [ ]:
# Exponenciación para calcular los odds ratios
odds_ratios = np.exp(coefficients)

print("Multiplicadores de los odds del modelo")
print("-----------------------")
for feature, odd in zip(["Ca", "Fe", "Mg"], odds_ratios):
    print(f"{feature}: {odd:>6.3f}")

Un multiplicador mayor que 1 eleva los odds, uno menor que 1 los reduce. Usa el coeficiente de Ca que obtuviste y el nombre de `model.classes_[1]` para explicar la dirección del cambio. Las otras entradas permanecen fijas en esa interpretación.

***

<a id="parte-3"></a>

### <span style="color:lightgreen">**Generación de mapas de probabilidades**</span>
***

Vamos a leer la predicción de una muestra y luego repetirla para cada píxel con las tres concentraciones. El orden de las columnas de `predict_proba` viene de `model.classes_`.

In [ ]:
x_muestra = X_test[:1]
z = model.decision_function(x_muestra)[0]
p = 1 / (1 + np.exp(-z))
probabilidades = model.predict_proba(x_muestra)[0]
print(dict(zip(model.classes_, probabilidades.round(3))))
print(f"Sigmoide del puntaje para {model.classes_[1]}: {p:.3f}")
print(f"Predicción: {y_pred[0]} | Etiqueta terrestre: {y_test[0]}")

Una prueba terrestre mide la separación de rocas terrestres. Para evaluar la transferencia habría que confirmar que las concentraciones elementales estén en la misma unidad y base química, comparar los rangos de las entradas y contrastar el mapa con referencias lunares. La resolución de los píxeles también puede mezclar distintas litologías.

In [ ]:
luna = np.stack([raster_Ca, raster_Fe, raster_Mg], axis=-1)
X_luna = luna[validos]
min_train, max_train = X_train.min(axis=0), X_train.max(axis=0)
fuera_rango = ((X_luna < min_train) | (X_luna > max_train)).any(axis=1)
rangos = pd.DataFrame(
    [min_train, max_train, X_luna.min(axis=0), X_luna.max(axis=0)],
    index=["Tierra mín.", "Tierra máx.", "Luna mín.", "Luna máx."],
    columns=["Ca", "Fe", "Mg"],
)
print(rangos.round(2))
print(f"Píxeles fuera de algún rango terrestre: {fuera_rango.mean():.1%}")

En los píxeles que superan algún rango, dejaremos el mapa sin estimación. Estar dentro de los tres rangos por separado tampoco demuestra que la combinación de Ca, Fe y Mg se parezca a una roca de entrenamiento.

In [ ]:
dentro = validos.copy()
dentro[validos] = ~fuera_rango
raster_anorth = np.full(raster_Ca.shape, np.nan)
indice_anorth = list(model.classes_).index("anorthosite")
if dentro.any():
    raster_anorth[dentro] = model.predict_proba(luna[dentro])[:, indice_anorth]

La máscara pasa de coordenadas de píxel a filas de una tabla al consultar el modelo y coloca cada resultado en su posición original. La escala de color muestra el puntaje de anortosita según el modelo terrestre balanceado.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(raster_anorth, extent=extension, cmap="viridis", vmin=0, vmax=1)
ax.set_facecolor("#dedede")
ax.set(title="Mare Crisium · estimación exploratoria de anortosita",
       xlabel="Longitud (°)", ylabel="Latitud (°)")
fig.colorbar(im, ax=ax, label="Probabilidad según el modelo terrestre")
plt.show()

Por último, guardaremos la figura en alta calidad y con el método `savefig`:

> El parámetro `bbox_inches="tight"` evita que el archivo exporte espacio en blanco innecesario.

In [ ]:
fig.savefig("litologia_lunar.png", dpi=300, bbox_inches="tight")

***

<a id="parte-4"></a>

### <span style="color:lightgreen">**En conclusión...**</span>

La regresión logística aprende un puntaje y la sigmoide lo convierte en una probabilidad estimada. Podemos reconstruir una fila y comprobar la correspondencia entre clases y columnas. Para interpretar el mapa como litología lunar hacen falta referencias independientes.

**Comprueba tu lectura.** ¿Qué pasaría con los colores si eliges la columna de basalto? ¿Qué datos pedirías para validar un límite entre unidades geológicas de la Luna?

***